# Módulo 19 · Práctica: inferencia rápida y barata

Harás que un mini-GPT genere **más rápido** (KV cache), **mejor** (top-k, top-p) y **más ligero** (int8, int4): las técnicas con que se sirven los LLM reales, en CPU. Completa las celdas `# TU CÓDIGO` y ejecuta la `# Comprobación` siguiente. Menos de 3 minutos en total.

| Parte | Qué haces |
| --- | --- |
| 0 | Datos y mini-GPT con cache |
| 1 | Entrenamiento exprés |
| 2 | KV cache y aceleración |
| 3 | Temperatura, top-k y top-p |
| 4 | Decodificación especulativa |
| 5 | Cuantización int8 e int4 |
| 6 | Retos |

## Parte 0 · Datos y modelo

Un texto corto escrito para el curso, repetido. Tokens = caracteres (módulo 15).

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_num_threads(1)   # mediciones de tiempo más estables

texto = ("en un lugar de la sierra vivia una vieja maestra que enseñaba a contar estrellas. "
 "cada noche subia al cerro con sus alumnos y les decia que la luz que veian habia salido hace miles de años. "
 "los niños preguntaban como podia una luz viajar tanto sin cansarse, y ella respondia que la luz no se cansa, solo se aleja. "
 "un dia un alumno le pregunto si las palabras tambien viajaban asi, y la maestra penso un rato antes de contestar. "
 "las palabras viajan de boca en boca y cambian un poco en cada viaje, dijo, por eso hay que cuidarlas como a las estrellas. ") * 3

chars = sorted(set(texto))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
codificar = lambda s: [stoi[c] for c in s]
decodificar = lambda l: "".join(itos[i] for i in l)
data = torch.tensor(codificar(texto))
corte = int(0.9 * len(data))
train_data, val_data = data[:corte], data[corte:]
print(V, "tokens de vocabulario,", len(data), "caracteres")

Es el mini-GPT del módulo 15, pero cada atención acepta un dict `cache` donde guarda K y V. Con cache, la consulta tiene `T` filas y las claves `Tk ≥ T`: la fila `i` ve hasta la columna `i + Tk − T`.

**Ejercicio 1.** Completa `actualizar_cache`.

In [ ]:
def actualizar_cache(cache, k, v):
    """k, v: (B, h, T_nuevo, d_head). Concatena a lo guardado en dim=2,
    guarda en cache['k'], cache['v'] y devuelve (k_total, v_total)."""
    # TU CÓDIGO
    return k, v

In [ ]:
class Atencion(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dh = h, d // h
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)

    def forward(self, x, cache=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, self.dh).transpose(1, 2) for t in (q, k, v)]
        if cache is not None:
            k, v = actualizar_cache(cache, k, v)
        Tk = k.shape[2]
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.dh)
        mascara = torch.ones(T, Tk, dtype=torch.bool).tril(diagonal=Tk - T)
        att = att.masked_fill(~mascara, float("-inf")).softmax(-1)
        y = (att @ v).transpose(1, 2).reshape(B, T, C)
        return self.proj(y)

class Bloque(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.att = Atencion(d, h)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x, cache=None):
        x = x + self.att(self.ln1(x), cache)
        return x + self.mlp(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self, V, d=128, h=4, L=4, ctx=320):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(ctx, d)
        self.bloques = nn.ModuleList([Bloque(d, h) for _ in range(L)])
        self.lnf, self.head = nn.LayerNorm(d), nn.Linear(d, V)

    def forward(self, idx, caches=None, inicio=0):
        T = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(inicio, inicio + T))
        for i, b in enumerate(self.bloques):
            x = b(x, None if caches is None else caches[i])
        return self.head(self.lnf(x))

modelo = MiniGPT(V)
print(f"{sum(p.numel() for p in modelo.parameters()):,} parámetros".replace(",", "."))

In [ ]:
# Comprobación
_c = {}
_k1, _v1 = torch.randn(1, 4, 3, 32), torch.randn(1, 4, 3, 32)
_k2, _v2 = torch.randn(1, 4, 1, 32), torch.randn(1, 4, 1, 32)
actualizar_cache(_c, _k1, _v1)
_k, _v = actualizar_cache(_c, _k2, _v2)
assert _k.shape == (1, 4, 4, 32), f"Tras 3 + 1 tokens la cache debe tener 4 posiciones, tiene {_k.shape}"
assert torch.equal(_k[:, :, 3:], _k2) and torch.equal(_v[:, :, :3], _v1), "Concatena en dim=2, lo viejo primero"
assert _c["k"] is _k, "Guarda el resultado en cache['k']"
print("Correcto")

## Parte 1 · Entrenamiento exprés

120 iteraciones de AdamW: no buscamos un gran modelo, solo distribuciones con sentido.

In [ ]:
def lote(datos, B=16, T=64):
    ix = torch.randint(0, len(datos) - T - 1, (B,))
    x = torch.stack([datos[i:i + T] for i in ix])
    y = torch.stack([datos[i + 1:i + T + 1] for i in ix])
    return x, y

opt = torch.optim.AdamW(modelo.parameters(), lr=3e-3)
t0 = time.time()
for it in range(120):
    x, y = lote(train_data)
    perdida = F.cross_entropy(modelo(x).view(-1, V), y.view(-1))
    opt.zero_grad(); perdida.backward(); opt.step()
    if it % 30 == 0:
        print(f"iter {it:3d}  pérdida {perdida.item():.3f}")
print(f"pérdida final {perdida.item():.3f} en {time.time() - t0:.1f} s")
modelo.eval();

## Parte 2 · KV cache

Sin cache, cada paso reprocesa **todo** el texto (coste total ∝ T²). Con cache, el prefill procesa el prompt y cada paso, **un solo token**.

**Ejercicio 2.** Completa `generar_con_cache` (greedy). Debe dar los mismos tokens que sin cache.

In [ ]:
@torch.no_grad()
def generar_sin_cache(modelo, idx, n):
    for _ in range(n):
        logits = modelo(idx)[:, -1]
        idx = torch.cat([idx, logits.argmax(-1, keepdim=True)], dim=1)
    return idx

prompt = torch.tensor([codificar("en un lugar")])
print(decodificar(generar_sin_cache(modelo, prompt, 60)[0].tolist()))

In [ ]:
@torch.no_grad()
def generar_con_cache(modelo, idx, n):
    """1) prefill: una pasada con todo idx y las caches;
    2) en cada paso, pasa SOLO el token nuevo con inicio=pos."""
    caches = [{} for _ in modelo.bloques]
    # TU CÓDIGO
    return idx

In [ ]:
# Comprobación
_a = generar_sin_cache(modelo, prompt, 80)
_b = generar_con_cache(modelo, prompt, 80)
assert _b.shape == _a.shape, f"Debe devolver {_a.shape[1]} tokens, devuelve {_b.shape[1]}"
assert torch.equal(_a, _b), "La cache no debe cambiar el resultado: revisa posiciones (inicio) y máscara"
print("Correcto:", decodificar(_b[0].tolist())[:80])

**Ejercicio 3.** Según 2 · L · n_kv · d_head · bytes = 2 · 4 · 4 · 32 · 4, cada token ocupa 4.096 bytes. Compruébalo: suma los bytes de todas las `k` y `v` de las caches y divide entre los tokens del prompt (`t.numel() * t.element_size()`).

In [ ]:
caches = [{} for _ in modelo.bloques]
with torch.no_grad():
    modelo(prompt, caches)
n_tok = prompt.shape[1]
# TU CÓDIGO
bytes_por_token = None

In [ ]:
# Comprobación
L, n_kv, d_head, nbytes = 4, 4, 32, 4
assert bytes_por_token == 2 * L * n_kv * d_head * nbytes, f"Esperaba {2*L*n_kv*d_head*nbytes}, obtengo {bytes_por_token}"
print("Correcto:", bytes_por_token, "bytes por token")

In [ ]:
def cronometrar(f, n, rep=2):
    mejor = float("inf")
    for _ in range(rep):
        t0 = time.perf_counter(); f(modelo, prompt, n); mejor = min(mejor, time.perf_counter() - t0)
    return mejor

print(" n   sin cache   con cache   aceleración")
aceleraciones = {}
for n in [50, 100, 200]:
    t_sin, t_con = cronometrar(generar_sin_cache, n), cronometrar(generar_con_cache, n)
    aceleraciones[n] = t_sin / t_con
    print(f"{n:3d}   {t_sin:7.3f} s   {t_con:7.3f} s   {aceleraciones[n]:5.1f}x")

In [ ]:
# Comprobación
assert aceleraciones[200] > 1.5, "Con 200 tokens la cache debería acelerar claramente"
assert aceleraciones[200] > aceleraciones[50], "La ventaja debe crecer con la longitud"
print("Correcto")

**Pregunta.** ¿Por qué la aceleración crece con n? Sin cache el paso t procesa unos 11 + t tokens; con cache, uno.

## Parte 3 · Muestreo

**Ejercicio 4.** Implementa `filtrar_top_k` y `filtrar_top_p`: los tokens eliminados quedan con logit `-inf`.

In [ ]:
def filtrar_top_k(logits, k):
    """Copia de logits (1D) con -inf fuera de los k mayores."""
    # TU CÓDIGO
    return logits

In [ ]:
def filtrar_top_p(logits, p):
    """Copia de logits (1D) con -inf fuera del menor conjunto de
    tokens más probables con probabilidad acumulada >= p."""
    # TU CÓDIGO
    return logits

In [ ]:
# Comprobación (ejemplo de la sección 19.4)
z = torch.tensor([2., 1., 0., -1.])
pk = filtrar_top_k(z, 2).softmax(-1)
assert torch.allclose(pk, torch.tensor([0.7311, 0.2689, 0., 0.]), atol=1e-4), f"top-k mal: {pk}"
pp = filtrar_top_p(z, 0.9).softmax(-1)
assert torch.allclose(pp, torch.tensor([0.6652, 0.2447, 0.0900, 0.]), atol=1e-4), f"top-p 0,9 mal: {pp}"
assert (filtrar_top_p(z, 0.7).softmax(-1) > 0).sum() == 2, "Con p = 0,7 deben quedar 2 tokens"
assert (filtrar_top_p(z, 0.5).softmax(-1) > 0).sum() == 1, "Con p = 0,5 debe quedar 1 token"
assert torch.equal(z, torch.tensor([2., 1., 0., -1.])), "No modifiques los logits originales"
print("Correcto")

Compara configuraciones: T = 2 sin filtro produce basura; top-p la limpia.

In [ ]:
@torch.no_grad()
def generar(modelo, idx, n, T=1.0, k=None, p=None, semilla=0):
    gen = torch.Generator().manual_seed(semilla)
    caches = [{} for _ in modelo.bloques]
    logits = modelo(idx, caches)[0, -1]
    pos = idx.shape[1]
    for _ in range(n):
        z = logits / T
        if k is not None: z = filtrar_top_k(z, k)
        if p is not None: z = filtrar_top_p(z, p)
        nuevo = torch.multinomial(z.softmax(-1), 1, generator=gen).view(1, 1)
        idx = torch.cat([idx, nuevo], dim=1)
        logits = modelo(nuevo, caches, inicio=pos)[0, -1]; pos += 1
    return decodificar(idx[0].tolist())

for cfg in [dict(T=1.0), dict(T=2.0), dict(T=2.0, p=0.8), dict(T=1.0, k=3), dict(T=0.5)]:
    print(cfg, "->", generar(modelo, prompt, 80, **cfg)[11:])

## Parte 4 · Decodificación especulativa

**Ejercicio 5.** Implementa la aceptación-rechazo de un token y comprueba por Monte Carlo que la salida sigue **exactamente** p.

In [ ]:
def paso_especulativo(p, q, gen):
    """x ~ q; acepta con prob. min(1, p[x]/q[x]); si no, muestrea de
    max(0, p - q) normalizada. Usa generator=gen. Devuelve un int."""
    # TU CÓDIGO
    return 0

In [ ]:
# Comprobación con p y q de la sección 19.5
p = torch.tensor([0.5, 0.3, 0.2]); q = torch.tensor([0.2, 0.6, 0.2])
gen = torch.Generator().manual_seed(1)
N = 20000
cuentas = torch.bincount(torch.tensor([paso_especulativo(p, q, gen) for _ in range(N)]), minlength=3)
frec = cuentas / N
print("frecuencias:", frec.numpy().round(3), " objetivo p:", p.numpy())
assert (frec - p).abs().max() < 0.015, "La salida debe seguir p, no q"
print("Correcto")

## Parte 5 · Cuantización

**Ejercicio 6.** Cuantización afín y su inversa (ejemplo de 19.6).

$$s = \frac{x_{\max}-x_{\min}}{2^b-1},\quad z=\mathrm{round}\!\left(-\frac{x_{\min}}{s}\right),\quad q=\mathrm{clip}\big(\mathrm{round}(x/s)+z,\,0,\,2^b-1\big),\quad \hat x = s\,(q-z)$$

In [ ]:
def cuantizar_asim(x, bits):
    """Devuelve (q, s, z): q en [0, 2^bits-1] (float), s float, z int."""
    # TU CÓDIGO
    return None, None, None

def decuantizar_asim(q, s, z):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
x = torch.tensor([-0.8, -0.1, 0.25, 1.6])
q8, s8, z8 = cuantizar_asim(x, 8)
assert z8 == 85 and torch.equal(q8, torch.tensor([0., 74., 112., 255.])), f"int8: z={z8}, q={q8}"
q4, s4, z4 = cuantizar_asim(x, 4)
assert z4 == 5 and torch.equal(q4, torch.tensor([0., 4., 7., 15.])), f"int4: z={z4}, q={q4}"
for q_, s_, z_ in [(q8, s8, z8), (q4, s4, z4)]:
    err = (decuantizar_asim(q_, s_, z_) - x).abs().max().item()
    assert err <= s_ / 2 + 1e-6, "El error no puede superar s/2"
    print(f"s = {s_:.6f}  error máximo = {err:.5f}  cota s/2 = {s_/2:.5f}")
print("Correcto")

**Ejercicio 7.** En pesos se usa cuantización **simétrica por filas**: un atípico solo estropea su fila.

In [ ]:
def cuantizar_filas(W, bits):
    """Simétrica, una escala por fila: s = max|fila| / (2^(bits-1)-1),
    q = round(W/s) en [-qmax, qmax]. s con forma (out, 1); q int8 si bits=8."""
    # TU CÓDIGO
    return None, None

In [ ]:
# Comprobación
W = modelo.bloques[0].mlp[0].weight.detach()
q, s = cuantizar_filas(W, 8)
assert q.dtype == torch.int8 and q.shape == W.shape and s.shape == (W.shape[0], 1), "Formas o tipo incorrectos"
assert q.abs().max() <= 127, "int8 simétrico va de -127 a 127"
err = (q.float() * s - W).abs()
assert (err <= s / 2 + 1e-7).all(), "Cada error debe ser <= s_fila / 2"
q4, s4 = cuantizar_filas(W, 4)
assert q4.abs().max() <= 7, "int4 simétrico va de -7 a 7"
print("Correcto")

Cuantizamos y decuantizamos (*fake quantization*) todas las capas lineales y medimos error, pérdida y acuerdo del argmax.

In [ ]:
import copy

@torch.no_grad()
def cuantizar_modelo(modelo, bits):
    m = copy.deepcopy(modelo)
    for mod in m.modules():
        if isinstance(mod, nn.Linear):
            q, s = cuantizar_filas(mod.weight, bits)
            mod.weight.copy_(q.float() * s)
    return m

@torch.no_grad()
def evaluar(m, ref):
    x = val_data[:-1].view(1, -1)[:, :300]; y = val_data[1:].view(1, -1)[:, :300]
    lg, lg_ref = m(x), ref(x)
    perdida = F.cross_entropy(lg.view(-1, V), y.view(-1)).item()
    acuerdo = (lg.argmax(-1) == lg_ref.argmax(-1)).float().mean().item()
    return perdida, acuerdo

def error_rel(m):
    num = den = 0.0
    for (n1, a), (n2, b) in zip(modelo.named_parameters(), m.named_parameters()):
        if a.dim() == 2 and "weight" in n1 and "tok" not in n1 and "pos" not in n1:
            num += (a - b).pow(2).sum().item(); den += a.pow(2).sum().item()
    return math.sqrt(num / den)

n_lin = sum(mod.weight.numel() for mod in modelo.modules() if isinstance(mod, nn.Linear))
resultados = {}
print("formato  MB pesos lineales  error rel.  pérdida val.  acuerdo argmax")
for nombre, bits in [("fp32", 32), ("int8", 8), ("int4", 4)]:
    m = modelo if bits == 32 else cuantizar_modelo(modelo, bits)
    perd, acu = evaluar(m, modelo)
    er = 0.0 if bits == 32 else error_rel(m)
    resultados[nombre] = (er, perd, acu)
    print(f"{nombre:6s}   {n_lin * bits / 8 / 1e6:8.3f}           {er:.4f}      {perd:.4f}       {acu:.3f}")

In [ ]:
# Comprobación
assert resultados["int8"][0] < 0.02, "El error relativo int8 por filas debería ser < 2 %"
assert resultados["int4"][0] > 5 * resultados["int8"][0], "int4 tiene ~16 veces menos niveles: su error es mucho mayor"
assert resultados["int8"][2] > 0.97, "int8 casi no debería cambiar el token más probable"
print("Correcto")

int8 casi no cambia las predicciones; el error de int4 es unas 18 veces mayor. Por eso int4 usa grupos de 64–128 pesos o GPTQ/AWQ.

## Parte 6 · Retos

1. Multiplica por 50 un peso de `W` y compara int8 por filas con una única escala global.
2. int4 con una escala cada 32 pesos: ¿error? ¿bits por peso con escalas fp16?
3. Borrador `cuantizar_modelo(modelo, 4)`, objetivo `modelo`: mide β medio sobre un texto.
4. Beam search con b = 3: compara su log-probabilidad con la de greedy.

## Soluciones de referencia

In [ ]:
# --- cache ---
# def actualizar_cache(cache, k, v):
#     if "k" in cache:
#         k = torch.cat([cache["k"], k], dim=2)
#         v = torch.cat([cache["v"], v], dim=2)
#     cache["k"], cache["v"] = k, v
#     return k, v

# --- gen ---
# @torch.no_grad()
# def generar_con_cache(modelo, idx, n):
#     caches = [{} for _ in modelo.bloques]
#     logits = modelo(idx, caches, inicio=0)[:, -1]
#     pos = idx.shape[1]
#     for _ in range(n):
#         nuevo = logits.argmax(-1, keepdim=True)
#         idx = torch.cat([idx, nuevo], dim=1)
#         logits = modelo(nuevo, caches, inicio=pos)[:, -1]
#         pos += 1
#     return idx

# --- topk ---
# def filtrar_top_k(logits, k):
#     umbral = torch.topk(logits, k).values[-1]
#     return logits.masked_fill(logits < umbral, float("-inf"))

# --- topp ---
# def filtrar_top_p(logits, p):
#     probs = logits.softmax(-1)
#     orden = probs.argsort(descending=True)
#     acum = probs[orden].cumsum(0)
#     n = int((acum < p).sum()) + 1          # tokens necesarios para llegar a p
#     fuera = orden[n:]
#     out = logits.clone()
#     out[fuera] = float("-inf")
#     return out

# --- spec ---
# def paso_especulativo(p, q, gen):
#     x = int(torch.multinomial(q, 1, generator=gen))
#     if torch.rand(1, generator=gen).item() < min(1.0, (p[x] / q[x]).item()):
#         return x
#     r = (p - q).clamp(min=0)
#     return int(torch.multinomial(r / r.sum(), 1, generator=gen))

# --- asim ---
# def cuantizar_asim(x, bits):
#     qmax = 2 ** bits - 1
#     s = (x.max() - x.min()).item() / qmax
#     z = int(round(-x.min().item() / s))
#     q = (torch.round(x / s) + z).clamp(0, qmax)
#     return q, s, z
#
# def decuantizar_asim(q, s, z):
#     return s * (q - z)

# --- sim ---
# def cuantizar_filas(W, bits):
#     qmax = 2 ** (bits - 1) - 1
#     s = W.abs().amax(dim=1, keepdim=True) / qmax
#     q = torch.round(W / s).clamp(-qmax, qmax)
#     if bits == 8:
#         q = q.to(torch.int8)
#     return q, s

# --- bytes por token (ejercicio 3) ---
# bytes_por_token = sum(t.numel() * t.element_size() for c in caches for t in (c["k"], c["v"])) / n_tok